# 01 — Dataset Analysis

Phase 1. Measures the dataset **as it actually is on disk** — no folder
names, counts or dimensions are assumed.

> The defect labels in this dataset are **synthetic** (defects painted onto
> real OK castings by script). See `PROJECT_DECISIONS.md` §1.1.


In [ ]:
# --- Colab setup (skip if running locally) ---
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('casting-defect-vlm'):
        # Replace with your repository URL, or upload the folder to Colab.
        raise SystemExit('Upload the casting-defect-vlm project folder to Colab first.')
    %cd casting-defect-vlm
    !pip install -q -r requirements.txt

sys.path.insert(0, os.path.abspath('..' if os.path.basename(os.getcwd())=='notebooks' else '.'))
print('python', sys.version.split()[0], '| colab:', IN_COLAB)


## 1. Download the dataset

Credentials come from environment variables — never hard-code them.


In [ ]:
# On Colab, prefer the Secrets panel (key icon) over pasting keys into a cell.
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!python scripts/download_dataset.py --which primary


## 2. Run the analysis


In [ ]:
!python scripts/analyze_dataset.py --data-dir data/raw --out-dir results/dataset_analysis


## 3. Inspect the summary


In [ ]:
import json, pandas as pd
from pathlib import Path

summary = json.loads(Path('results/dataset_analysis/dataset_summary.json').read_text())
print('Root            :', summary['dataset_root'])
print('Valid images    :', summary['n_valid_images'])
print('Corrupted       :', summary['n_corrupted_images'])
print('Classes         :', summary['n_classes'])
print('Label type      :', summary['label_type'])
print('Channels        :', summary['channel_distribution'])
print('Dimensions      :', summary['most_common_dimensions'][:3])


## 4. Class distribution


In [ ]:
dist = pd.DataFrame({
    'images': summary['images_per_class'],
    'percent': summary['class_percentages'],
}).sort_values('images', ascending=False)
dist


### Class imbalance check

With 11 defect folders and 1 OK folder, expect a heavy skew toward
*defective*. A model answering 'Defective' every time would score high
accuracy while being useless — so macro-F1 and the confusion matrix
matter more than accuracy here.


In [ ]:
ok_like = [c for c in summary['images_per_class'] if c.lower() in {'ok','ok_front','good'}]
n_ok = sum(summary['images_per_class'][c] for c in ok_like)
n_def = summary['n_valid_images'] - n_ok
print(f'OK        : {n_ok}')
print(f'Defective : {n_def}')
print(f'Majority-class baseline accuracy = {max(n_ok, n_def)/summary["n_valid_images"]:.4f}')


## 5. Leakage and duplicate check


In [ ]:
dup = summary['duplicates']
for k, v in dup.items():
    if isinstance(v, int):
        print(f'{k:<38}{v}')


## 6. Plots


In [ ]:
from IPython.display import Image, display
for name in ['class_distribution.png', 'image_dimensions.png', 'sample_images.png']:
    p = Path('results/dataset_analysis') / name
    if p.exists():
        print(name); display(Image(str(p)))


## 7. Image metadata


In [ ]:
meta = pd.read_csv('results/dataset_analysis/image_metadata.csv')
print(meta.shape)
meta.head()


## 8. Findings

_Record what the numbers above actually show. Do not write a number here
that was not produced by the analysis._
